In [220]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler,OrdinalEncoder,StandardScaler
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.tree import DecisionTreeRegressor,ExtraTreeRegressor
from sklearn.model_selection import train_test_split,KFold,cross_val_score
from sklearn.metrics import r2_score,mean_absolute_error
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.svm import SVR
import pickle
from sklearn.neighbors import KNeighborsRegressor
pd.set_option('display.max_rows', None)

In [221]:
df = pd.read_csv('/content/pakwheels_website.csv')

In [222]:
df.head()

,year,Km_travelled,fuel_type,city,assembly,engine,body_type,interior_scores,exterior_scores,comfort_scores,company,price
0,2020,50000,Petrol,Sindh,Imported,2000,Crossover,high,high,high,KIA,0.64
1,2013,90000,Petrol,Islamabad,Imported,1800,Sedan,medium,high,high,Mercedes-Benz,0.97
2,2011,94000,Petrol,other,Imported,2100,Sedan,medium,medium,high,Mercedes-Benz,0.92
3,2018,135000,Hybrid,Sindh,Imported,1500,Hatchback,medium,low,low,Toyota,0.52
4,2021,62000,Petrol,other,Local,2000,Sedan,medium,medium,high,Hyundai,0.56


In [223]:
X = df.drop(columns=['price'])
y = np.log1p(df['price'])

In [224]:
y.head()

,price
0,0.494696
1,0.678034
2,0.652325
3,0.418710
4,0.444686


In [225]:
y.min()

0.009950330853168083

In [226]:
y.max()

2.740840023925201

In [227]:
transformer = ColumnTransformer(transformers=[
    ('scaling',StandardScaler(),['year','Km_travelled','engine']),
    ('ordinal_e',OrdinalEncoder(categories=[['low','medium','high'], ['low','medium','high'], ['low','medium','high']]),['interior_scores','exterior_scores','comfort_scores']),
    ('ohe',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),['fuel_type','city','assembly','body_type','company'])
],remainder='passthrough')

In [228]:
model = {
    'lr' : LinearRegression(),
    'dtree' :  DecisionTreeRegressor(),
    'lass' : Lasso(),
    'ridge' : Ridge(),
    'extra_tree' : ExtraTreeRegressor(),
    'random_forest' : RandomForestRegressor(),
    'gradient_boost' : GradientBoostingRegressor(),
    'svr' : SVR(),
    'knn' : KNeighborsRegressor()
}

In [229]:
scores = []
mean_error = []
model_name = []

In [230]:
def calculate_score(model):
  pipeline = Pipeline(steps=[
      ('preprocess',transformer),
      ('model',model)
  ])

  X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42)
  pipeline.fit(X_train,y_train)
  y_pred = pipeline.predict(X_test)
  score = r2_score(y_test,y_pred)
  mean_abs_error = mean_absolute_error(np.expm1(y_test),np.expm1(y_pred))
  scores.append(score)
  mean_error.append(mean_abs_error)

In [231]:
for x,i in model.items():
  calculate_score(i)
  model_name.append(x)

In [232]:
model_perform = pd.DataFrame({
    'model_name' : model_name,
    'score' : scores,
    'mae' : mean_error,
})

In [233]:
pipeline = Pipeline(steps=[
  ('preprocess',transformer),
  ('model',GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,

    max_depth=4
))
])

X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.5)
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scaling', StandardScaler(),
                                                  ['year', 'Km_travelled',
                                                   'engine']),
                                                 ('ordinal_e',
                                                  OrdinalEncoder(categories=[['low',
                                                                              'medium',
                                                                              'high'],
                                                                             ['low',
                                                                              'medium',
                                                                              'high'],
                                                                             ['low',
                                                                              'medium',
                                                                              'high']]),
                                                  ['interior_scores',
                                                   'exterior_scores',
                                                   'comfort_scores']),
                                                 ('ohe',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['fuel_type', 'city',
                                                   'assembly', 'body_type',
                                                   'company'])])),
                ('model',
                 GradientBoostingRegressor(learning_rate=0.05, max_depth=4,
                                           n_estimators=300))])

In [234]:
y_pred = pipeline.predict(X_test)

In [235]:
r2_score(y_test,y_pred)

0.9634221909492251

In [237]:
test_data = [

# 1 New 1200cc sedan perfect condition
[2024, 5000, 'Petrol', 'Punjab', 'Local', 1200, 'Sedan',
 'high','high','high','Toyota'],

# 2 2018 1800cc but terrible condition
[2018, 140000, 'Petrol', 'Punjab', 'Local', 1800, 'Sedan',
 'low','low','low','Toyota'],

# 3 2016 2500cc Prado high km
[2016, 180000, 'Petrol', 'Punjab', 'Imported', 2500, 'SUV',
 'medium','medium','medium','Toyota'],

# 4 2023 2000cc Imported Audi
[2023, 15000, 'Petrol', 'Punjab', 'Imported', 2000, 'Sedan',
 'high','high','high','Audi'],

# 5 2015 1500cc BMW imported
[2015, 110000, 'Petrol', 'Punjab', 'Imported', 1500, 'Sedan',
 'high','high','high','BMW'],

# 6 2022 660cc Alto average
[2022, 30000, 'Petrol', 'Punjab', 'Local', 660, 'Hatchback',
 'medium','medium','medium','Suzuki'],

# 7 2019 3500cc Lexus
[2019, 90000, 'Petrol', 'Punjab', 'Imported', 3500, 'SUV',
 'high','high','high','Lexus'],

# 8 2020 Hybrid Corolla
[2020, 60000, 'Hybrid', 'Punjab', 'Imported', 1800, 'Sedan',
 'high','high','high','Toyota'],

# 9 2014 4000cc Land Cruiser old
[2014, 200000, 'Petrol', 'Punjab', 'Imported', 4000, 'SUV',
 'medium','medium','medium','Toyota'],

# 10 2024 1500cc Chinese SUV
[2024, 2000, 'Petrol', 'Punjab', 'Local', 1500, 'SUV',
 'high','high','high','Changan'],

# 11 2017 1000cc but perfect condition
[2017, 60000, 'Petrol', 'Punjab', 'Local', 1000, 'Hatchback',
 'high','high','high','Suzuki'],

# 12 2021 2200cc Diesel SUV
[2021, 50000, 'Diesel', 'Punjab', 'Imported', 2200, 'SUV',
 'high','high','high','Toyota'],

# 13 2013 3000cc Mercedes
[2013, 150000, 'Petrol', 'Punjab', 'Imported', 3000, 'Sedan',
 'high','high','high','Mercedes-Benz'],

# 14 2023 800cc cheap hatchback
[2023, 10000, 'Petrol', 'Punjab', 'Local', 800, 'Hatchback',
 'medium','medium','medium','Suzuki'],

# 15 2012 1800cc Civic
[2012, 170000, 'Petrol', 'Punjab', 'Local', 1800, 'Sedan',
 'medium','medium','medium','Honda'],

# 16 2024 6200cc AMG extreme
[2024, 1000, 'Petrol', 'Punjab', 'Imported', 6200, 'SUV',
 'high','high','high','Mercedes-Benz'],

# 17 2020 1300cc but imported assembly
[2020, 45000, 'Petrol', 'Punjab', 'Imported', 1300, 'Sedan',
 'medium','medium','medium','Toyota'],

# 18 2019 2000cc SUV very high km
[2019, 210000, 'Petrol', 'Punjab', 'Local', 2000, 'SUV',
 'medium','medium','medium','Toyota'],

]

df_test = pd.DataFrame(test_data,columns=X.columns)

In [238]:
np.expm1(pipeline.predict(df_test))

array([0.56901263, 0.50742773, 1.32736862, 1.53293043, 0.64109314,
       0.30946345, 3.87602645, 0.82963696, 2.07456641, 0.74720433,
       0.4074712 , 1.29899441, 1.58773734, 0.2725927 , 0.34923766,
       7.57971986, 0.49429635, 0.68867831])

In [239]:
mean_absolute_error(np.expm1(y_test),np.expm1(y_pred))

0.06302991162910371

In [240]:
# feat_importance.sort_values(by='importance',ascending=False)

In [241]:
pickle.dump(pipeline,open('price_model.pkl','wb'))

In [242]:
pickle.dump(X,open('df.pkl','wb'))